# Ch.12 — Testing AI Systems: Catching the 14% Wrong-Order Bug

**Track**: AI (03-ai) | **Chapter**: 12 | **Grand Challenge**: Mamma Rosa's PizzaBot  
**Previous**: [ch11_advanced_agentic_patterns](../ch11_advanced_agentic_patterns/) | **Next**: Production deployment

---

The CEO says PizzaBot has 0 bugs. Production says 14% of orders are wrong.

This notebook is the diagnostic. You'll write tests that:
1. Reproduce the 14% wrong-order rate
2. Isolate the bug to the retrieval layer
3. Write unit, integration, model, and property-based tests that prevent it recurring
4. Wire everything into a GitHub Actions CI gate

All code is runnable. No hardcoded credentials — use environment variables.

## Section 1 — Setup & Imports

Install `pytest`, `pytest-cov`, and `hypothesis`. We also define a minimal PizzaBot stub so all cells run without real API calls — the stub is swapped for the real client in integration tests.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## Section 2 — The Broken PizzaBot: Reproducing the 14% Bug

**Failure-first.** Before writing a single fix, you need a test that names the failure exactly.

Operations hands you 200 production logs. You replay them against the pipeline and collect `(query, expected_price, actual_price)` triples. The wrong-order rate is 14%.

The next cell reproduces this in miniature — no real API needed.

In [ ]:
# TODO: Implement this cell


## Section 3 — Unit Testing: Model Input/Output Assertions

Unit tests isolate one function. They don't call real APIs — dependencies are mocked at the boundary.

**What we're testing per component:**
- `format_prompt()` — correct template + no injection
- `embed()` — returns a list of floats with the right length
- `retrieve()` — returns `List[Document]`, respects metadata filter
- `parse_order_response()` — extracts price as a float, item as a string

Run these with `%%ipytest` — equivalent to `pytest tests/unit/`.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## Section 4 — Reproducibility Tests

LLM outputs are probabilistic by default. `temperature=0` makes them deterministic — same input, same output, every time. If your tests assert exact strings, you need this property. If they fail randomly, your `temperature` is non-zero.

The pattern: call the same query twice, assert `response_1 == response_2`. Use `pytest.mark.parametrize` to cover multiple prompts.

In [ ]:
# TODO: Implement this cell


## Section 5 — Integration Testing: RAG Pipeline (Ingestion → Retrieval → Generation)

Integration tests fire the **real pipeline** — no mocks. Each stage is tested in isolation so a failure points to exactly one component.

**Three stages:**
1. **Ingestion** — are all docs in the index with correct metadata?
2. **Retrieval** — does a query return the right document at rank 1?  ← where the 14% bug lives
3. **Generation** — does the final response reference the retrieved document's price?

Testing stage 2 in isolation is what localises the bug.

In [ ]:
# TODO: Implement this cell


## Section 6 — Model Tests: Shape, Invariance & Directional Expectations

Inspired by Ribeiro et al. (ACL 2020) — the CheckList paper. Three types:

| Type | Question | Example |
|------|----------|---------|
| **Shape** | Is the output the right structure? | Response is a non-empty string, ≤500 chars, contains a price |
| **Invariance (INV)** | Does irrelevant variation break output? | LARGE PEPPERONI == large pepperoni (same price) |
| **Directional (DIR)** | Does relevant variation change output monotonically? | 2 pizzas > 1 pizza in price |

Shape tests catch type errors. Invariance tests catch NLP fragility. Directional tests catch pricing logic bugs.

In [ ]:
# TODO: Implement this cell


## Section 7 — Property-Based Testing with hypothesis

Unit tests check specific inputs you thought of. `hypothesis` generates inputs you didn't.

It works by:
1. You define a *strategy* (`st.sampled_from`, `st.integers`, etc.)
2. `hypothesis` generates 100–200 random inputs from that strategy
3. If a test fails, it *shrinks* the input to the smallest counterexample

The PizzaBot pricing function has a monotonicity property: more pizzas → higher price, always. `hypothesis` tries to break it by generating random `(size, quantity)` pairs.

In [ ]:
# TODO: Implement this cell


## Section 8 — Measuring Coverage with pytest-cov

Coverage tells you which lines were *executed* during tests. It's a floor, not a ceiling — a line can be covered by a test that asserts nothing.

**Target: ≥ 80% for production modules.** Below 80% means large untested branches.

Below we simulate a coverage report for our stub modules, then show how to close a gap.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## Section 9 — CI/CD: GitHub Actions Workflow

The tests you've written are only valuable if they run on every pull request. The GitHub Actions workflow below:

1. **Triggers** on push to `main` and any pull request
2. **Unit + model tests**: run on every PR (free, fast, no API cost)
3. **Integration tests**: run on push to `main` only (costs ~$0.05 per run)
4. **Blocks merge** if any test fails

`OPENAI_API_KEY` is injected from GitHub Secrets — never stored in code or YAML.

In [ ]:
# TODO: Implement this cell


## Section 10 — Running the Full Suite

All tests combined. After fixing the retrieval bug, every test should be GREEN and coverage ≥ 80%.

The command below uses `subprocess` to show the full pytest output inline in the notebook.

In [ ]:
# TODO: Implement this cell


---

## Summary

| Section | Tests written | What it caught |
|---------|--------------|----------------|
| Setup | — | Environment & stubs configured |
| Bug reproduction | Harness (wrong_order_rate) | 14% wrong-order rate surfaced |
| Unit tests | 14 pytest tests | Component contract violations |
| Reproducibility | 6 parametrized tests | Non-determinism at temperature > 0 |
| Integration tests | 10 tests (3 stages) | **Retrieval bug: lunch menu returned for dinner queries** |
| Model tests | 12 tests (shape/INV/DIR) | Pricing monotonicity violations |
| Property tests | 4 hypothesis tests | Crashes on edge-case inputs |
| Coverage | 5 tests | Uncovered delivery-zone validation branch |
| CI/CD | GitHub Actions YAML | All tests run on every PR, blocks merge on failure |
| Full suite | Smoke + regression | 0 failures after bug fix ✅ |

**wrong_order_rate: 14% → 0% ✅**

**Next**: [DevOps Ch.4 — CI/CD Pipelines](../../07-devops_fundamentals/) — deploy the tested, CI-gated PizzaBot to production at 10,000 orders/day.